# Week 12 — LDA, QDA, Gaussian Naive Bayes และ KNN

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week12/`)
- `biz_customers.csv` — ลูกค้า 1203 คน

**ทุกข้อ**: fit ด้วย train = 960 แถวแรก, วัดผลกับ test = 243 แถวที่เหลือ (ไม่สับข้อมูล)

## โมเดลที่ใช้ — ต่างกันตรง "สมมติฐาน" ที่ใส่

| โมเดล | สมมติฐาน | รูปขอบเขตการตัดสิน |
|---|---|---|
| **LDA** | แต่ละคลาสมี **covariance เดียวกัน** | **เส้นตรง** (เลขหนึ่งคลุมทั้งชุด) |
| **QDA** | แต่ละคลาสมี covariance **ต่างกันได้** | **เส้นโค้ง** (วงรี/พาราโบลา) |
| **GaussianNB** | ตัวแปร **อิสระกันในคลาส** | สมมติว่าไม่มี interaction |
| **KNN** | ไม่มีสมมติฐาน ใช้ระยะทาง | เส้นตัดแบบ non-linear |

**เปรียบเทียบ LDA กับ QDA:** ถ้า QDA ไม่ดีกว่า LDA มักแปลว่าข้อมูลแทบจะเป็น Gaussian
และรูปทรงเดียวกันจริง ๆ (แบบนี้ LDA ซึ่งจำนวนพารามิเตอร์น้อยกว่า จะประมาณได้ดีกว่า)

## `predict` กับ `predict_proba` ต่างกันอย่างไร

- `predict(X)` → คลาสที่**ความน่าจะเป็นสูงสุด** (เทียบกันอยู่แล้ว ไม่ได้ใช้ threshold ที่เราเลือกเอง)
- `predict_proba(X)` → ความน่าจะเป็นของ**แต่ละคลาส** คอลัมน์ที่ 0 = คลาส 0, คอลัมน์ที่ 1 = คลาส 1
  → เอามาตั้ง threshold เองได้ (ข้อ 12.8)

## KNN + StandardScaler

KNN วัดระยะทาง → **หน่วยของตัวแปรมีผลมาก** ตัวแปรที่มีสเกลใหญ่จะครอบงำตัวอื่น
ต้อง standardize ก่อน และ **fit scaler บน train แล้วใช้ scaler ตัวเดียวกับ test**
→ ใช้ `make_pipeline(StandardScaler(), KNeighborsClassifier(...))` จะจัดการให้เอง

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week12')
NEEDED = ['biz_customers.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ไฟล์ครบแล้ว: data\week12


In [2]:
# --- ตรวจไฟล์ข้อมูล + เวอร์ชันเครื่องมือ (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd
import sklearn

DATA_DIR = Path(f'data/week12')
NEEDED = ['biz_customers.csv']

print('scikit-learn', sklearn.__version__, '| pandas', pd.__version__)
for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์')
    for col in ['responded', 'churn', 'loyalty_member']:
        print(f'    {col}: {df[col].value_counts().to_dict()}')

scikit-learn 1.9.0 | pandas 3.0.5
biz_customers.csv       1203 แถว x 13 คอลัมน์
    responded: {0: 832, 1: 371}
    churn: {0: 1060, 1: 143}
    loyalty_member: {0: 750, 1: 453}


## ข้อ 12.1 — ค่าเฉลี่ยของ web_visits ในคลาส 1 ที่ LDA ประมาณได้

- `lda.means_` มีรูป `(จำนวนคลาส, จำนวนตัวแปร)` → แถวที่ **1 = คลาส 1**
- ค่านี้คือ**ค่าเฉลี่ยของ train** ในคลาสนั้น ไม่ใช่ค่ากลางของ test

In [3]:
# --- ข้อ 12.1 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cust = pd.read_csv(Path('data/week12') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['email_opens', 'web_visits', 'income_k', 'age']
train = cust.iloc[:960]

lda = LinearDiscriminantAnalysis().fit(train[cols], train['responded'])
answer = float(lda.means_[1][cols.index('web_visits')])

print('รูปของ lda.means_ =', lda.means_.shape, '(แถว = คลาส, คอลัมน์ = ตัวแปรตามลำดับที่ใส่)')
print('ค่าเฉลี่ยคลาส 0 =', np.round(lda.means_[0], 4))
print('ค่าเฉลี่ยคลาส 1 =', np.round(lda.means_[1], 4))
print('ค่าเฉลี่ย web_visits ของคลาส 1 =', answer)
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

รูปของ lda.means_ = (2, 4) (แถว = คลาส, คอลัมน์ = ตัวแปรตามลำดับที่ใส่)
ค่าเฉลี่ยคลาส 0 = [ 4.9311  7.6931 30.0499 39.5973]
ค่าเฉลี่ยคลาส 1 = [ 5.911   9.1301 31.3336 36.5342]
ค่าเฉลี่ย web_visits ของคลาส 1 = 9.13013698630137
ค่าที่กรอก: 3 ตำแหน่ง -> 9.130


## ข้อ 12.2 — F1 ของ LDA ทำนายสมาชิกสะสมแต้ม

- ใช้ `predict` ตามค่าเริ่มต้มของ sklearn = เลือกคลาสที่ความน่าจะเป็นสูงสุด
- F1 = ค่าเดียวที่รวม precision กับ recall เข้าด้วยกัน

In [4]:
# --- ข้อ 12.2 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cust = pd.read_csv(Path('data/week12') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.metrics import f1_score, confusion_matrix

cols = ['web_visits', 'tenure_months', 'orders_90d', 'email_opens']
train, test = cust.iloc[:960], cust.iloc[960:]

lda = LinearDiscriminantAnalysis().fit(train[cols], train['loyalty_member'])
yhat = lda.predict(test[cols])
answer = float(f1_score(test['loyalty_member'], yhat, pos_label=1))

cm = confusion_matrix(test['loyalty_member'], yhat, labels=[0, 1])
tp, fp, fn = cm[1, 1], cm[0, 1], cm[1, 0]
print('confusion matrix:'); print(cm)
prec, rec = tp / (tp + fp), tp / (tp + fn)
print(f'TP={tp} FP={fp} FN={fn} -> precision = {prec:.4f}, recall = {rec:.4f}')
print('F1 =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

confusion matrix:
[[142  11]
 [ 40  50]]
TP=50 FP=11 FN=40 -> precision = 0.8197, recall = 0.5556
F1 = 0.6622516556291391
ค่าที่กรอก: 4 ตำแหน่ง -> 0.6623


## ข้อ 12.3 — ความน่าจะเป็นที่ QDA ให้กับลูกค้าใหม่

- ลูกค้าคนนี้**ไม่ได้อยู่ในไฟล์** → ต้องสร้าง DataFrame ใหม่ เรียงคอลัมน์ตามลำดับที่ fit
- `predict_proba(X)[0, 1]` = ความน่าจะเป็นของ**คลาส 1** (คอลัมน์ที่ 1 เสมอเมื่อคลาสเรียง 0, 1)

In [5]:
# --- ข้อ 12.3 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cust = pd.read_csv(Path('data/week12') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['email_opens', 'web_visits', 'income_k', 'age']
train = cust.iloc[:960]

qda = QuadraticDiscriminantAnalysis().fit(train[cols], train['responded'])

new = pd.DataFrame([[4, 10, 41.8, 28]], columns=cols)      # email, web, income, age
proba = qda.predict_proba(new)[0]
answer = float(proba[1])

print('ลูกค้าใหม่: email_opens = 4, web_visits = 10, income_k = 41.8, age = 28')
print('ความน่าจะเป็นคลาส 0 =', round(float(proba[0]), 6))
print('ความน่าจะเป็นคลาส 1 =', answer, '  <- คำตอบ')
print('ผล predict (คลาสที่ prob สูงสุด) =', int(qda.predict(new)[0]))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

ลูกค้าใหม่: email_opens = 4, web_visits = 10, income_k = 41.8, age = 28
ความน่าจะเป็นคลาส 0 = 0.726486
ความน่าจะเป็นคลาส 1 = 0.27351364187017313   <- คำตอบ
ผล predict (คลาสที่ prob สูงสุด) = 0
ค่าที่กรอก: 4 ตำแหน่ง -> 0.2735


## ข้อ 12.4 — Recall ของ Gaussian Naive Bayes

- GaussianNB สมมติว่า**ตัวแปรแต่ละตัวเป็นอิสระต่อกันในคลาส**
  ทำให้คำนวณเร็วมาก (ไม่ต้องคูณ matrix covariance)
- recall = TP / (TP + FN) = จับได้กี่ % ของคนที่ตอบรับจริง

In [6]:
# --- ข้อ 12.4 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cust = pd.read_csv(Path('data/week12') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.metrics import recall_score, confusion_matrix

cols = ['email_opens', 'web_visits', 'age']
train, test = cust.iloc[:960], cust.iloc[960:]

nb = GaussianNB().fit(train[cols], train['responded'])
yhat = nb.predict(test[cols])
answer = float(recall_score(test['responded'], yhat, pos_label=1))

cm = confusion_matrix(test['responded'], yhat, labels=[0, 1])
print('confusion matrix:'); print(cm)
print('ตอบรับจริงใน test =', int(test['responded'].sum()),
      '| ทำนายว่าตอบ =', int(yhat.sum()))
print('recall =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

confusion matrix:
[[149  15]
 [ 63  16]]
ตอบรับจริงใน test = 79 | ทำนายว่าตอบ = 31
recall = 0.20253164556962025
ค่าที่กรอก: 4 ตำแหน่ง -> 0.2025


## ข้อ 12.5 — LDA กับ QDA ทำนายไม่ตรงกันกี่คน

- fit ทั้งสองโมเดลด้วย train ชุดเดียวกัน ใช้ตัวแปรเดียวกัน
- เทียบ `predict` ทีละแถวของ test
- ยิ่งไม่ตรงกันน้อย = ข้อมูลเป็น Gaussian แบบเดียวกันจริง (กรณีนี้ LDA เพียงพอ)

In [7]:
# --- ข้อ 12.5 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cust = pd.read_csv(Path('data/week12') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['email_opens', 'income_k', 'age']
train, test = cust.iloc[:960], cust.iloc[960:]

lda = LinearDiscriminantAnalysis().fit(train[cols], train['responded'])
qda = QuadraticDiscriminantAnalysis().fit(train[cols], train['responded'])
p_lda = lda.predict(test[cols])
p_qda = qda.predict(test[cols])
diff = int((p_lda != p_qda).sum())
answer = diff

print('test =', len(test), 'คน')
print('LDA ทำนายเป็นคลาส 1 =', int(p_lda.sum()), 'คน')
print('QDA ทำนายเป็นคลาส 1 =', int(p_qda.sum()), 'คน')
print('ทำนายไม่ตรงกัน =', diff, 'คน | ตรงกัน =', len(test) - diff, 'คน')
print('(ยิ่งต่างน้อย = ข้อมูลเป็น Gaussian รูปเดียวกันจริง ๆ LDA ก็เพียงพอและใช้พารามิเตอร์น้อยกว่า)')
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

test = 243 คน
LDA ทำนายเป็นคลาส 1 = 7 คน
QDA ทำนายเป็นคลาส 1 = 16 คน
ทำนายไม่ตรงกัน = 9 คน | ตรงกัน = 234 คน
(ยิ่งต่างน้อย = ข้อมูลเป็น Gaussian รูปเดียวกันจริง ๆ LDA ก็เพียงพอและใช้พารามิเตอร์น้อยกว่า)
ค่าที่กรอก: จำนวนเต็ม -> 9


## ข้อ 12.6 — KNN (K=21) หลัง standardize: ความน่าจะเป็นเฉลี่ยของคลาส 1

- ใช้ `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=21))`
  → scaler ถูก fit บน train แล้วใช้กับ test โดยอัตโนมัติ (ไม่ leak ข้อมูล test)
- `predict_proba` ของ KNN = **สัดส่วนเพื่อนบ้าน K คนที่อยู่ในคลาส 1**
  ดังนั้นค่าเฉลี่ยจะเป็นตัวเลขที่คูณ 1/21 เสมอ

In [8]:
# --- ข้อ 12.6 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cust = pd.read_csv(Path('data/week12') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['email_opens', 'web_visits', 'income_k', 'spend_k']
train, test = cust.iloc[:960], cust.iloc[960:]

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=21))
pipe.fit(train[cols], train['responded'])

p = pipe.predict_proba(test[cols])[:, 1]
answer = float(p.mean())

print('K = 21 -> แต่ละค่าความน่าจะเป็นเป็นจำนวนเต็ม/21 เสมอ')
print('ค่าที่พบบ่อยที่สุด:', np.unique(np.round(p, 6)))
print('ความน่าจะเป็นเฉลี่ยของ 243 คน =', answer)
print('เทียบสัดส่วนผู้ตอบรับจริงใน test =', round(float(test['responded'].mean()), 6))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

K = 21 -> แต่ละค่าความน่าจะเป็นเป็นจำนวนเต็ม/21 เสมอ
ค่าที่พบบ่อยที่สุด: [0.       0.047619 0.095238 0.142857 0.190476 0.238095 0.285714 0.333333
 0.380952 0.428571 0.47619  0.52381  0.571429 0.619048 0.666667]
ความน่าจะเป็นเฉลี่ยของ 243 คน = 0.30001959631589264
เทียบสัดส่วนผู้ตอบรับจริงใน test = 0.325103
ค่าที่กรอก: 4 ตำแหน่ง -> 0.3000


## ข้อ 12.7 — สเกลของตัวแปรมีผลต่อ KNN จริงไหม

- โมเดลเดียวกัน (K=9) ตัวแปรเดียวกัน แต่ fit สองแบบ: ค่าดิบ กับ StandardScaler
- ตัวแปรที่มีหน่วยใหญ่ (เช่น `income_k`, `spend_k`) จะเพิ่มระยะยุคลิดไปเอง
  ทำให้ตัวแปรอื่นแทบไม่มีผลต่อการเลือกเพื่อนบ้าน

In [9]:
# --- ข้อ 12.7 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cust = pd.read_csv(Path('data/week12') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['email_opens', 'income_k', 'age', 'spend_k']
train, test = cust.iloc[:960], cust.iloc[960:]

raw = KNeighborsClassifier(n_neighbors=9).fit(train[cols], train['responded'])
scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9))
scaled.fit(train[cols], train['responded'])

a = raw.predict(test[cols])
b = scaled.predict(test[cols])
answer = int((a != b).sum())

print('ช่วงของตัวแปร (train):')
print(train[cols].agg(['min', 'max']).to_string())
print()
print('แบบที่ 1 (ค่าดิบ)      ทำนายเป็นคลาส 1 =', int(a.sum()), 'คน')
print('แบบที่ 2 (standardize) ทำนายเป็นคลาส 1 =', int(b.sum()), 'คน')
print('ทำนายไม่ตรงกัน =', answer, 'คน | ตรงกัน =', len(test) - answer, 'คน')
print('-> สเกลของตัวแปรมีผลจริง กับ KNN ต้อง standardize เสมอ')
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

ช่วงของตัวแปร (train):
     email_opens  income_k  age  spend_k
min            0       7.4   18     0.20
max           14     108.8   70     9.73

แบบที่ 1 (ค่าดิบ)      ทำนายเป็นคลาส 1 = 32 คน
แบบที่ 2 (standardize) ทำนายเป็นคลาส 1 = 30 คน
ทำนายไม่ตรงกัน = 42 คน | ตรงกัน = 201 คน
-> สเกลของตัวแปรมีผลจริง กับ KNN ต้อง standardize เสมอ
ค่าที่กรอก: จำนวนเต็ม -> 42


## ข้อ 12.8 — ลด threshold ของ LDA เพื่อจับคนเลิกซื้อให้ได้มากขึ้น

- LDA ให้ **ความน่าจะเป็น** ที่มัก "มั่นใจน้อย" (ค่าเฉลี่ยต่ำกว่าความจริง)
  ถ้าใช้ `predict` ตามปกติจะจับคนเลิกซื้อได้น้อยมาก
- วิธีแก้: ใช้ `predict_proba` แล้วตั้งเกณฑ์เองที่ **0.12**
- ผลคือ recall สูงขึ้นมาก แลกกับ precision ที่ลดลง

In [10]:
# --- ข้อ 12.8 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cust = pd.read_csv(Path('data/week12') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.metrics import recall_score, confusion_matrix

cols = ['tenure_months', 'web_visits', 'age']
train, test = cust.iloc[:960], cust.iloc[960:]

lda = LinearDiscriminantAnalysis().fit(train[cols], train['churn'])
p = lda.predict_proba(test[cols])[:, 1]

print('ค่าเฉลี่ยของ prob คลาส 1 =', round(float(p.mean()), 4),
      '| สูงสุด =', round(float(p.max()), 4), '(ต่ำกว่า 1 มาก เพราะ LDA ไม่ได้ calibrate)')

yhat_default = lda.predict(test[cols])                      # ใช้ predict ตามปกติ
rec_default = recall_score(test['churn'], yhat_default, pos_label=1)

yhat = (p >= 0.12).astype(int)
answer = float(recall_score(test['churn'], yhat, pos_label=1))

cm = confusion_matrix(test['churn'], yhat, labels=[0, 1])
print()
print('test: churn จริง =', int(test['churn'].sum()), 'คน')
print('ใช้ predict ปกติ (เทียบ prob สูงสุด): จับได้', int(yhat_default.sum()), 'คน -> recall =', round(float(rec_default), 4))
print('ใช้เกณฑ์ 0.12: ทำนายเป็นเสี่ยง', int(yhat.sum()), 'คน -> recall =', answer)
print('confusion matrix:'); print(cm)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

ค่าเฉลี่ยของ prob คลาส 1 = 0.1162 | สูงสุด = 0.4352 (ต่ำกว่า 1 มาก เพราะ LDA ไม่ได้ calibrate)

test: churn จริง = 28 คน
ใช้ predict ปกติ (เทียบ prob สูงสุด): จับได้ 0 คน -> recall = 0.0
ใช้เกณฑ์ 0.12: ทำนายเป็นเสี่ยง 95 คน -> recall = 0.6071428571428571
confusion matrix:
[[137  78]
 [ 11  17]]
ค่าที่กรอก: 4 ตำแหน่ง -> 0.6071


## ข้อ 12.9 — หา K ที่ให้ F1 สูงสุด (13 ค่า: 1, 3, 5, ..., 25)

- ทุกข้อใช้ StandardScaler ก่อน KNN เสมอ
- ลอง K = 1 ถึง 25 เฉพาะเลขคี่ แล้วเทียบ F1 บนชุด test
- ข้อสังเกต: F1 มัก **ลดลง** เมื่อ K ใหญ่ขึ้น ถ้าข้อมูลไม่มี noise มาก
  เพราะทั้ง train และ test มาจากการแจกแจงเดียวกัน

In [11]:
# --- ข้อ 12.9 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cust = pd.read_csv(Path('data/week12') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.metrics import f1_score

cols = ['email_opens', 'income_k', 'age', 'spend_k']
train, test = cust.iloc[:960], cust.iloc[960:]

rows = []
for k in range(1, 26, 2):                       # 1, 3, 5, ..., 25
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    pipe.fit(train[cols], train['responded'])
    f1 = f1_score(test['responded'], pipe.predict(test[cols]), pos_label=1)
    rows.append((k, float(f1)))
    print(f'  K = {k:<3} F1 = {f1:.6f}')

best_k, answer = max(rows, key=lambda r: r[1])
print()
print('K ที่ให้ F1 สูงสุด =', best_k, '| F1 =', round(answer, 6))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

  K = 1   F1 = 0.397260
  K = 3   F1 = 0.330579
  K = 5   F1 = 0.330579
  K = 7   F1 = 0.280702
  K = 9   F1 = 0.275229
  K = 11  F1 = 0.247619
  K = 13  F1 = 0.183673
  K = 15  F1 = 0.189474
  K = 17  F1 = 0.170213
  K = 19  F1 = 0.170213
  K = 21  F1 = 0.107527
  K = 23  F1 = 0.107527


  K = 25  F1 = 0.133333

K ที่ให้ F1 สูงสุด = 1 | F1 = 0.39726
ค่าที่กรอก: 4 ตำแหน่ง -> 0.3973


## ข้อ 12.10 — AUC ของ QDA

- **AUC วัดคุณภาพของการจัดอันดับ** ไม่ต้องเลือก threshold
  - AUC = 0.5 = เดาเสมอ, 1.0 = สมบูรณ์แบบ
- ต้องใช้ **ความน่าจะเป็นของคลาส 1** เป็นคะแนนของแต่ละคน
- ใช้ `roc_auc_score(y_true, proba)`

In [12]:
# --- ข้อ 12.10 ---
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cust = pd.read_csv(Path('data/week12') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.metrics import roc_auc_score

cols = ['email_opens', 'web_visits', 'income_k', 'age']
train, test = cust.iloc[:960], cust.iloc[960:]

qda = QuadraticDiscriminantAnalysis().fit(train[cols], train['responded'])
p = qda.predict_proba(test[cols])[:, 1]
answer = float(roc_auc_score(test['responded'], p))

print('test =', len(test), 'คน | ตอบรับจริง =', int(test['responded'].sum()))
print('ความน่าจะเป็นคลาส 1: ต่ำสุด =', round(float(p.min()), 4),
      '| เฉลี่ย =', round(float(p.mean()), 4), '| สูงสุด =', round(float(p.max()), 4))
print('AUC =', answer, '(0.5 = สุ่ม, 1.0 = สมบูรณ์แบบ)')
print('เทียบ: ตอบรับจริงมีค่าเฉลี่ย', round(float(p[test['responded'] == 1].mean()), 4),
      '| ไม่ตอบมีค่าเฉลี่ย', round(float(p[test['responded'] == 0].mean()), 4),
      '-> คนกลุ่มแรกค่าสูงกว่า = จัดอันดับได้ดี')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

test =

 243 คน | ตอบรับจริง = 79
ความน่าจะเป็นคลาส 1: ต่ำสุด = 0.1197 | เฉลี่ย = 0.3003 | สูงสุด = 0.8433
AUC = 0.6907224451991355 (0.5 = สุ่ม, 1.0 = สมบูรณ์แบบ)
เทียบ: ตอบรับจริงมีค่าเฉลี่ย 0.3643 | ไม่ตอบมีค่าเฉลี่ย 0.2695 -> คนกลุ่มแรกค่าสูงกว่า = จัดอันดับได้ดี
ค่าที่กรอก: 4 ตำแหน่ง -> 0.6907


## สรุปคำตอบทั้ง 10 ข้อ (Week 12)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อนที่เว็บยอมรับ`

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 12.1 | LDA means_ ของ web_visits ในคลาส 1 | **9.130** | 3 ตำแหน่ง | 0.0015 |
| 12.2 | F1 ของ LDA ทำนาย loyalty_member | **0.6623** | 4 ตำแหน่ง | 0.00015 |
| 12.3 | QDA predict_proba ของลูกค้าใหม่ | **0.2735** | 4 ตำแหน่ง | 0.00015 |
| 12.4 | recall ของ GaussianNB | **0.2025** | 4 ตำแหน่ง | 0.00015 |
| 12.5 | จำนวนที่ LDA กับ QDA ทำนายไม่ตรง | **9** | จำนวนเต็ม | 0.5 |
| 12.6 | KNN K=21 ค่าเฉลี่ย prob คลาส 1 | **0.3000** | 4 ตำแหน่ง | 0.00015 |
| 12.7 | KNN K=9 ค่าดิบ vs scaled ทำนายไม่ตรง | **42** | จำนวนเต็ม | 0.5 |
| 12.8 | recall ของ LDA ที่ threshold 0.12 | **0.6071** | 4 ตำแหน่ง | 0.00015 |
| 12.9 | F1 สูงสุดใน 13 ค่า K | **0.3973** | 4 ตำแหน่ง | 0.00015 |
| 12.10 | AUC ของ QDA | **0.6907** | 4 ตำแหน่ง | 0.00015 |

### สรุปสั้น ๆ เรื่อง "ทำไม"

- **12.5** LDA กับ QDA ทำนายตรงกัน **234 จาก 243 คน** → ข้อมูลเกือบเป็น Gaussian รูปเดียวกัน
  LDA ซึ่งใช้พารามิเตอร์น้อยกว่าเพียงพอและเสถียรกว่า
- **12.7** สเกลมีผลจริง: ค่าดิบกับ standardized ทำนายต่างกัน **42 คนจาก 243 (17%)**
- **12.8** ค่าเฉลี่ย prob ของ LDA ต่ำมาก (เพราะไม่ได้ calibrate) ลดเกณฑ์เป็น 0.12
  ทำให้ recall ขึ้นจาก **0.0357 → 0.6071** แลกกับการทำนายเป็นเสี่ยงมากขึ้น
- **12.9** F1 ลดลงตาม K ที่เพิ่มขึ้น (0.397 → 0.13) → ข้อมูลไม่มี noise มาก **K เล็กดีกว่า** คือ K = 1

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน